# 01 — Historical archive

The competition exposes a historical archive so you can experiment **offline**:
one gzip-JSONL file per `event_type` × `quarter`, each line a past event with its
disclosure. This notebook:

1. Browses the archive manifest (`GET /archive`)
2. Downloads and **caches** the files locally (`data/archive/`, gitignored)
3. Loads them into a pandas DataFrame
4. Inspects a single event and its disclosure

With a live key you pull the real archive; without one, you load the small bundled
sample so the mechanics are still clear.

> Download URLs are short-lived and signed. The helper refreshes an expired URL
> automatically (via `GET /archive/{event_type}/{quarter}`) when you pass it a
> client.

In [ ]:
from pathlib import Path

from IPython.display import display

from examples import Client, load_config
from examples.archive import download_archive, load_archive
from examples.frames import manifest_frame

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "sample").is_dir())
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache
SAMPLE_DIR = REPO / "data" / "sample"  # tiny bundled stand-in

config = load_config()
print("Mode:", "LIVE" if config.is_live else "SAMPLE (no EM_API_KEY — using bundled data)")

## 1. Browse the manifest

Each row is one downloadable file — what event type and quarter it covers, how
many events it holds, and how big it is. (The signed URLs are omitted from this
view.)

In [ ]:
manifest = None
if config.is_live:
    with Client.from_env() as client:
        manifest = client.archive_manifest()
    display(manifest_frame(manifest))
else:
    print("Sample mode: no live manifest. We'll load the bundled sample archive below.")

## 2. Download & cache

`download_archive` writes each file into `data/archive/`, **skipping** any file
already cached at the size the manifest reports — so re-running is cheap. Narrow
the pull with `only=`, e.g. `only=lambda f: f.quarter >= "2026Q1"`.

In [ ]:
if config.is_live:
    with Client.from_env() as client:
        paths = download_archive(
            manifest,
            ARCHIVE_DIR,
            client=client,  # lets it refresh expired URLs
            # only=lambda f: f.event_type == "EARNINGS_RELEASE",
        )
    source = ARCHIVE_DIR
    print(f"{len(paths)} file(s) cached in {ARCHIVE_DIR}")
else:
    source = SAMPLE_DIR
    print(f"Sample mode: reading the bundled archive in {SAMPLE_DIR}")

## 3. Load into pandas

`load_archive` reads every `*.jsonl.gz` under a directory (or a single file) into
one DataFrame, adding `event_type` and `quarter` provenance columns from each
filename.

In [ ]:
df = load_archive(source)
print(f"Loaded {len(df):,} events")
print("Columns:", list(df.columns))
df.head()

## 4. Inspect one event

Pull a single row and read its focal assets and disclosure facts. (The bundled
sample inlines disclosures under a `disclosure` field; the live archive may carry
additional fields — inspect `df.columns` to see what a given dataset provides.)

In [ ]:
row = df.iloc[0]

focal = row.get("focal_assets") if "focal_assets" in df.columns else None
tickers = [a["identifier_value"] for a in (focal or [])]
print(f"event_type={row['event_type']}  quarter={row['quarter']}  tickers={tickers}")
print(f"event_datetime={row.get('event_datetime')}\n")

disclosure = row.get("disclosure") if "disclosure" in df.columns else None
if isinstance(disclosure, dict):
    for item in disclosure.get("items", []):
        if item.get("kind") == "facts":
            print(f"facts from {item.get('source')!r}:")
            for fact in item["content"]:
                print("  -", fact)
else:
    print("No inlined 'disclosure' column here — check df.columns for this dataset's fields.")

## Where to go from here

The archive gives you the **inputs** — events and their disclosures. To turn this
into a backtest you'd:

1. Pair each event with the realized next-day reaction for its focal asset, from
   your own market-data source.
2. Convert that reaction into a percentile against the asset's historical
   distribution — the same `[0, 1]` target the competition scores.
3. Score your model's predicted percentile against it offline, and iterate.

When your model is ready, move it into a deployed webhook handler (see the starter
repos) — that's what receives live events and submits predictions. These notebooks
deliberately stop short of `POST /predictions`.